# Analytics Marts

## Overview

This notebook transforms the normalized movie database into business-ready analytical datasets.

The Analytics Marts are designed to provide simplified, analysis-ready views of the underlying relational database while preserving data integrity and clearly defined table grain.

The marts will be constructed using SQL through SQLAlchemy and loaded into Pandas for validation and downstream analysis.

## Objectives

- Transform normalized source tables into analytical datasets.
- Define the grain of every analytical mart.
- Resolve many-to-many relationships safely.
- Prevent join multiplication and duplicate records.
- Aggregate descriptive attributes where appropriate.
- Validate primary keys and row counts.
- Produce reusable datasets for exploratory analysis and visualization.

## Analytical Marts

The following marts will be created:

1. Artist Diary
2. Movie Master
3. Movie Production
4. Movie Keywords
5. Movie Languages
6. Movie Genre
7. Artist Performance
8. Movie Financial Performance

## Design Principle

Every mart must have a clearly defined grain.

Before creating a mart, we must answer:

> What does one row represent?

The mart will only be considered valid when its intended grain is maintained.

In [1]:
pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Importing the required libraries
import pandas as pd
from  sqlalchemy import create_engine
from pathlib import Path

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
# Creating a database engine
db_path = Path.home() / "Documents" / "Movie DB SQL Analytics" / "Database" / "TMDB-a-4006.db"

engine = create_engine(f"sqlite:///{db_path}")

print("Database Connection established successfully!")

Database Connection established successfully!


## Analytics Mart Architecture

### Source Layer

Normalized relational tables:

actors
casts
movies
genres
genremap
keywords
keywordmap
languages
languagemap
productioncompanies
productioncompanymap
productioncountries
productioncountrymap
oscars

### Analytics Layer

                ┌─────────────────┐
                │     MOVIES      │
                └────────┬────────┘
                         │
       ┌─────────────────┼─────────────────┐
       │                 │                 │
       ▼                 ▼                 ▼
 Artist Diary      Movie Master      Movie Financial
       │                 │
       │       ┌─────────┼─────────┐
       │       ▼         ▼         ▼
       │    Genres   Languages  Keywords
       │
       ▼
 Artist Performance

       │
       ▼
 Movie Production

In [ ]:
"""
Mart 1 — Movie Master
Grain:
One row = one movie
This should be our foundational mart.
The important point is that we aggregate all many-to-many attributes before joining them to movies.
"""

movie_master_query = """
SELECT
    m.movie_id AS Movie_ID,
    m.title AS Movie_Title,
    m.release_date AS Release_Date,
    m.popularity AS Popularity,
    m.budget AS Budget,
    m.revenue AS Revenue,

    g.Genres,
    k.Keywords,
    l.Languages,
    pc.Production_Companies,
    pco.Production_Countries

FROM movies m

LEFT JOIN (
    SELECT
        gm.movie_id,
        GROUP_CONCAT(DISTINCT g.genre_name) AS Genres
    FROM genremap gm
    LEFT JOIN genres g
        ON gm.genre_id = g.genre_id
    GROUP BY gm.movie_id
) g
    ON m.movie_id = g.movie_id

LEFT JOIN (
    SELECT
        km.movie_id,
        GROUP_CONCAT(DISTINCT k.keyword_name) AS Keywords
    FROM keywordmap km
    LEFT JOIN keywords k
        ON km.keyword_id = k.keyword_id
    GROUP BY km.movie_id
) k
    ON m.movie_id = k.movie_id

LEFT JOIN (
    SELECT
        lm.movie_id,
        GROUP_CONCAT(DISTINCT l.language_name) AS Languages
    FROM languagemap lm
    LEFT JOIN languages l
        ON lm.ISO_639_1 = l.ISO_639_1
    GROUP BY lm.movie_id
) l
    ON m.movie_id = l.movie_id

LEFT JOIN (
    SELECT
        pcm.movie_id,
        GROUP_CONCAT(DISTINCT pc.production_company_name)
            AS Production_Companies
    FROM productioncompanymap pcm
    LEFT JOIN productioncompanies pc
        ON pcm.production_company_id = pc.production_company_id
    GROUP BY pcm.movie_id
) pc
    ON m.movie_id = pc.movie_id

LEFT JOIN (
    SELECT
        pcom.movie_id,
        GROUP_CONCAT(DISTINCT pco.production_country_name)
            AS Production_Countries
    FROM productioncountrymap pcom
    LEFT JOIN productioncountries pco
        ON pcom.ISO_3166_1 = pco.ISO_3166_1
    GROUP BY pcom.movie_id
) pco
    ON m.movie_id = pco.movie_id;
"""

movie_master = pd.read_sql(movie_master_query, engine)

movie_master.head()

,Movie_ID,Movie_Title,Release_Date,Popularity,Budget,Revenue,Genres,Keywords,Languages,Production_Companies,Production_Countries
0,5,Four Rooms,1995-12-09 00:00:00.000000,22.876230,4000000,4300000.0,"Comedy,Crime","hotel,new year's eve,witch,bet,hotel room,sper...",English,"Miramax Films,A Band Apart",United States of America
1,11,Star Wars,1977-05-25 00:00:00.000000,126.393695,11000000,775398007.0,"Adventure,Action,Science Fiction","android,galaxy,hermit,death star,lightsaber,je...",English,"Lucasfilm,Twentieth Century Fox Film Corporation",United States of America
2,12,Finding Nemo,2003-05-30 00:00:00.000000,85.688789,94000000,940335536.0,"Animation,Family","father son relationship,harbor,underwater,fish...",English,Pixar Animation Studios,United States of America
3,13,Forrest Gump,1994-07-06 00:00:00.000000,138.133331,55000000,677945399.0,"Drama,Comedy,Romance","vietnam veteran,hippie,mentally disabled,runni...",English,Paramount Pictures,United States of America
4,14,American Beauty,1999-09-15 00:00:00.000000,80.878605,15000000,356296601.0,Drama,"male nudity,female nudity,adultery,midlife cri...",English,"DreamWorks SKG,Jinks/Cohen Company",United States of America


In [50]:
movie_master.to_csv(r"C:\Users\user\Documents\Movie DB SQL Analytics\CSV files\movie_master.csv", index=False)

In [6]:
# Row Count
movie_master.shape

(4803, 11)

In [7]:
# Unique Movies
movie_master["Movie_ID"].nunique()

4803

In [8]:
# Duplicate Check
movie_master["Movie_ID"].duplicated().sum()

np.int64(0)

In [9]:
# Validation
assert movie_master["Movie_ID"].is_unique

In [ ]:
"""
Mart 2 — Movie Production
This mart should be more granular than Movie Master.
Grain:
One row = one Movie × Production Company × Production Country combination
However, there is an important design decision here.
For a production mart, I recommend keeping the relationships separately rather than forcing companies and countries into one grain.
"""

movie_production_company_query = """
SELECT
    m.movie_id AS Movie_ID,
    m.title AS Movie_Title,
    pcm.production_company_id AS Production_Company_ID,
    pc.production_company_name AS Production_Company

FROM movies m
INNER JOIN productioncompanymap pcm
    ON m.movie_id = pcm.movie_id

LEFT JOIN productioncompanies pc
    ON pcm.production_company_id = pc.production_company_id;
"""

movie_production_company = pd.read_sql(
    movie_production_company_query,
    engine
)

movie_production_company.head()

,Movie_ID,Movie_Title,Production_Company_ID,Production_Company
0,11,Star Wars,1,Lucasfilm
1,85,Raiders of the Lost Ark,1,Lucasfilm
2,87,Indiana Jones and the Temple of Doom,1,Lucasfilm
3,89,Indiana Jones and the Last Crusade,1,Lucasfilm
4,217,Indiana Jones and the Kingdom of the Crystal S...,1,Lucasfilm


In [51]:
movie_production_company.to_csv(r"C:\Users\user\Documents\Movie DB SQL Analytics\CSV files\movie_production_company.csv", index=False)

In [12]:
movie_production_company.duplicated(
    subset=["Movie_ID", "Production_Company_ID"]
).sum()

np.int64(0)

In [13]:
"""
Production Country Mart
Grain:
One row = one Movie × Production Country
"""

movie_production_country_query = """
SELECT
    m.movie_id AS Movie_ID,
    m.title AS Movie_Title,
    pcom.ISO_3166_1 AS Country_Code,
    pco.production_country_name AS Production_Country

FROM movies m
INNER JOIN productioncountrymap pcom
    ON m.movie_id = pcom.movie_id

LEFT JOIN productioncountries pco
    ON pcom.ISO_3166_1 = pco.ISO_3166_1;
"""

movie_production_country = pd.read_sql(
    movie_production_country_query,
    engine
)

movie_production_country.head()

,Movie_ID,Movie_Title,Country_Code,Production_Country
0,5,Four Rooms,US,United States of America
1,11,Star Wars,US,United States of America
2,12,Finding Nemo,US,United States of America
3,13,Forrest Gump,US,United States of America
4,14,American Beauty,US,United States of America


In [52]:
movie_production_country.to_csv(r"C:\Users\user\Documents\Movie DB SQL Analytics\CSV files\movie_production_country.csv", index=False)

In [14]:
movie_production_company.duplicated(
    subset=["Movie_ID", "Production_Company_ID"]
).sum()

np.int64(0)

In [17]:
"""
Mart 3 — Movie Keywords
Grain:
One row = one Movie × Keyword
"""

movie_keywords_query = """
SELECT
    m.movie_id AS Movie_ID,
    m.title AS Movie_Title,
    k.keyword_id AS Keyword_ID,
    k.keyword_name AS Keyword

FROM movies m

INNER JOIN keywordmap km
    ON m.movie_id = km.movie_id

LEFT JOIN keywords k
    ON km.keyword_id = k.keyword_id;
"""

movie_keywords = pd.read_sql(
    movie_keywords_query,
    engine
)

movie_keywords.head()

,Movie_ID,Movie_Title,Keyword_ID,Keyword
0,16,Dancer in the Dark,30,individual
1,65,8 Mile,30,individual
2,77,Memento,30,individual
3,197,Braveheart,30,individual
4,207,Dead Poets Society,30,individual


In [53]:
movie_keywords.to_csv(r"C:\Users\user\Documents\Movie DB SQL Analytics\CSV files\movie_keywords.csv", index=False)

In [18]:
movie_keywords.duplicated(
    subset=["Movie_ID", "Keyword_ID"]
).sum()

np.int64(0)

In [15]:
"""
Mart 4 — Movie Languages
Grain:
One row = one Movie × Language
"""
movie_languages_query = """
SELECT
    m.movie_id AS Movie_ID,
    m.title AS Movie_Title,
    lm.ISO_639_1 AS Language_Code,
    l.language_name AS Language

FROM movies m

INNER JOIN languagemap lm
    ON m.movie_id = lm.movie_id

LEFT JOIN languages l
    ON lm.ISO_639_1 = l.ISO_639_1;
"""

movie_languages = pd.read_sql(
    movie_languages_query,
    engine
)

movie_languages.head()

,Movie_ID,Movie_Title,Language_Code,Language
0,868,Tsotsi,af,Afrikaans
1,1123,Catch a Fire,af,Afrikaans
2,1372,Blood Diamond,af,Afrikaans
3,17654,District 9,af,Afrikaans
4,22600,Gangster's Paradise: Jerusalema,af,Afrikaans


In [54]:
movie_languages.to_csv(r"C:\Users\user\Documents\Movie DB SQL Analytics\CSV files\movie_languages.csv", index=False)

In [16]:
movie_languages.duplicated(
    subset=["Movie_ID", "Language_Code"]
).sum()

np.int64(0)

In [19]:
"""
Mart 5 — Movie Genres
Grain:
One row = one Movie × Genre"""

movie_genres_query = """
SELECT
    m.movie_id AS Movie_ID,
    m.title AS Movie_Title,
    g.genre_id AS Genre_ID,
    g.genre_name AS Genre

FROM movies m

INNER JOIN genremap gm
    ON m.movie_id = gm.movie_id

LEFT JOIN genres g
    ON gm.genre_id = g.genre_id;
"""

movie_genres = pd.read_sql(
    movie_genres_query,
    engine
)

movie_genres.head()

,Movie_ID,Movie_Title,Genre_ID,Genre
0,11,Star Wars,12,Adventure
1,18,The Fifth Element,12,Adventure
2,22,Pirates of the Caribbean: The Curse of the Bla...,12,Adventure
3,58,Pirates of the Caribbean: Dead Man's Chest,12,Adventure
4,62,2001: A Space Odyssey,12,Adventure


In [55]:
movie_genres.to_csv(r"C:\Users\user\Documents\Movie DB SQL Analytics\CSV files\movie_genres.csv", index=False)

In [20]:
movie_genres.duplicated(
    subset=["Movie_ID", "Genre_ID"]
).sum()

np.int64(0)

In [23]:
"""
Mart 6 — Artist Diary
This is where we correct the earlier artist_diary approach.
Grain:
One row = one Artist × Movie × Character
We should not directly join all many-to-many dimensions.
Instead, aggregate movie-level attributes first.
"""

artist_diary_query = """
SELECT
    a.actor_id AS Artist_ID,
    a.actor_name AS Artist_Name,
    a.gender AS Gender,
    c.characters AS Character_Name,
    m.movie_id AS Movie_ID,
    m.title AS Movie_Title,
    m.release_date AS Release_Date,
    m.popularity AS Popularity,
    m.budget AS Budget,
    m.revenue AS Revenue,
    g.Genres,
    l.Languages,
    pc.Production_Companies,
    pco.Production_Countries

FROM actors a

INNER JOIN casts c
    ON a.actor_id = c.actor_id

INNER JOIN movies m
    ON c.movie_id = m.movie_id

LEFT JOIN (
    SELECT
        gm.movie_id,
        GROUP_CONCAT(DISTINCT g.genre_name) AS Genres
    FROM genremap gm
    LEFT JOIN genres g
        ON gm.genre_id = g.genre_id
    GROUP BY gm.movie_id
) g
    ON m.movie_id = g.movie_id

LEFT JOIN (
    SELECT
        lm.movie_id,
        GROUP_CONCAT(DISTINCT l.language_name) AS Languages
    FROM languagemap lm
    LEFT JOIN languages l
        ON lm.ISO_639_1 = l.ISO_639_1
    GROUP BY lm.movie_id
) l
    ON m.movie_id = l.movie_id

LEFT JOIN (
    SELECT
        pcm.movie_id,
        GROUP_CONCAT(DISTINCT pc.production_company_name)
            AS Production_Companies
    FROM productioncompanymap pcm
    LEFT JOIN productioncompanies pc
        ON pcm.production_company_id = pc.production_company_id
    GROUP BY pcm.movie_id
) pc
    ON m.movie_id = pc.movie_id

LEFT JOIN (
    SELECT
        pcom.movie_id,
        GROUP_CONCAT(DISTINCT pco.production_country_name)
            AS Production_Countries
    FROM productioncountrymap pcom
    LEFT JOIN productioncountries pco
        ON pcom.ISO_3166_1 = pco.ISO_3166_1
    GROUP BY pcom.movie_id
) pco
    ON m.movie_id = pco.movie_id

ORDER BY
    m.release_date;
"""

artist_diary = pd.read_sql(
    artist_diary_query,
    engine
)

artist_diary.head()

,Artist_ID,Artist_Name,Gender,Character_Name,Movie_ID,Movie_Title,Release_Date,Popularity,Budget,Revenue,Genres,Languages,Production_Companies,Production_Countries
0,1496,Tod Browning,2,Crook (uncredited),3059,Intolerance,1916-09-04 00:00:00.000000,3.232447,385907,8394751.0,Drama,English,"Triangle Film Corporation,Wark Producing Corp.",United States of America
1,3247,Noble Johnson,2,Babylonian Soldier (uncredited),3059,Intolerance,1916-09-04 00:00:00.000000,3.232447,385907,8394751.0,Drama,English,"Triangle Film Corporation,Wark Producing Corp.",United States of America
2,8630,Erich von Stroheim,0,Second Pharisee (uncredited),3059,Intolerance,1916-09-04 00:00:00.000000,3.232447,385907,8394751.0,Drama,English,"Triangle Film Corporation,Wark Producing Corp.",United States of America
3,8728,Eugene Pallette,2,Prosper Latour (French Story),3059,Intolerance,1916-09-04 00:00:00.000000,3.232447,385907,8394751.0,Drama,English,"Triangle Film Corporation,Wark Producing Corp.",United States of America
4,8820,Joseph Henabery,2,L'amiral de Coligny / Defendant,3059,Intolerance,1916-09-04 00:00:00.000000,3.232447,385907,8394751.0,Drama,English,"Triangle Film Corporation,Wark Producing Corp.",United States of America


In [49]:
artist_diary.to_csv(r"C:\Users\user\Documents\Movie DB SQL Analytics\CSV files\artist_diary.csv", index=False)

In [27]:
artist_diary.duplicated(
    subset=[
        "Artist_ID",
        "Movie_ID",
        "Character_Name"
    ]
).sum()

np.int64(0)

In [28]:
artist_diary.shape

(106257, 14)

In [29]:
""""
Artist Performance Mart
Now we can build an aggregated artist-level mart.
Grain:
One row = one Artist
"""
artist_performance_query = """
SELECT
    a.actor_id AS Artist_ID,
    a.actor_name AS Artist_Name,

    COUNT(DISTINCT c.movie_id) AS Movie_Count,

    SUM(DISTINCT m.revenue) AS Total_Revenue,

    AVG(DISTINCT m.popularity) AS Avg_Popularity,

    SUM(DISTINCT m.budget) AS Total_Budget

FROM actors a

LEFT JOIN casts c
    ON a.actor_id = c.actor_id

LEFT JOIN movies m
    ON c.movie_id = m.movie_id

GROUP BY
    a.actor_id,
    a.actor_name;
"""

artist_performance = pd.read_sql(
    artist_performance_query,
    engine
)

artist_performance.head()

,Artist_ID,Artist_Name,Movie_Count,Total_Revenue,Avg_Popularity,Total_Budget
0,1,George Lucas,4,1.603064e+09,41.266890,261000000
1,2,Mark Hamill,4,1.920286e+09,68.321404,83350000
2,3,Harrison Ford,32,6.735114e+09,40.118679,1306227000
3,4,Carrie Fisher,16,2.741736e+09,29.589817,334350000
4,5,Peter Cushing,1,7.753980e+08,126.393695,11000000


In [56]:
artist_performance.to_csv(r"C:\Users\user\Documents\Movie DB SQL Analytics\CSV files\artist_performance.csv", index=False)

In [30]:
"""
Movie Financial Performance
Grain:
One row = one Movie
"""

movie_financial_query = """
SELECT
    movie_id AS Movie_ID,
    title AS Movie_Title,
    budget AS Budget,
    revenue AS Revenue,

    revenue - budget AS Profit,

    CASE
        WHEN budget > 0
        THEN ROUND(
            CAST(revenue - budget AS FLOAT) / budget * 100,
            2
        )
        ELSE NULL
    END AS ROI_Percentage

FROM movies;
"""

movie_financial = pd.read_sql(
    movie_financial_query,
    engine
)

movie_financial.head()

,Movie_ID,Movie_Title,Budget,Revenue,Profit,ROI_Percentage
0,5,Four Rooms,4000000,4300000.0,300000.0,7.50
1,11,Star Wars,11000000,775398007.0,764398007.0,6949.07
2,12,Finding Nemo,94000000,940335536.0,846335536.0,900.36
3,13,Forrest Gump,55000000,677945399.0,622945399.0,1132.63
4,14,American Beauty,15000000,356296601.0,341296601.0,2275.31


In [57]:
movie_financial.to_csv(r"C:\Users\user\Documents\Movie DB SQL Analytics\CSV files\movie_financial.csv", index=False)

In [32]:
# Mart Validation Framework
def validate_mart(df, key_columns, mart_name):

    print(f"========== {mart_name} ==========")

    print(f"Rows: {len(df):,}")

    print(
        f"Columns: {len(df.columns):,}"
    )

    print(
        f"Duplicate Grain Records: "
        f"{df.duplicated(subset=key_columns).sum():,}"
    )

    print(
        f"Null Key Records: "
        f"{df[key_columns].isnull().any(axis=1).sum():,}"
    )

    print(
        f"Unique Grain Records: "
        f"{df[key_columns].drop_duplicates().shape[0]:,}"
    )

    print()

In [33]:
validate_mart(
    movie_master,
    ["Movie_ID"],
    "Movie Master"
)

========== Movie Master ==========
Rows: 4,803
Columns: 11
Duplicate Grain Records: 0
Null Key Records: 0
Unique Grain Records: 4,803



In [34]:
validate_mart(
    artist_diary,
    ["Artist_ID", "Movie_ID", "Character_Name"],
    "Artist Diary"
)

========== Artist Diary ==========
Rows: 106,257
Columns: 14
Duplicate Grain Records: 0
Null Key Records: 0
Unique Grain Records: 106,257



In [42]:
validate_mart(
    movie_keywords,
    ["Movie_ID", "Keyword_ID"],
    "Movie Keywords"
)

========== Movie Keywords ==========
Rows: 36,194
Columns: 4
Duplicate Grain Records: 0
Null Key Records: 0
Unique Grain Records: 36,194



In [35]:
validate_mart(
    movie_languages,
    ["Movie_ID", "Language_Code"],
    "Movie Languages"
)

========== Movie Languages ==========
Rows: 6,937
Columns: 4
Duplicate Grain Records: 0
Null Key Records: 0
Unique Grain Records: 6,937



In [37]:
validate_mart(
    movie_genres,
    ["Movie_ID", "Genre_ID"],
    "Movie Genres"
)

========== Movie Genres ==========
Rows: 12,160
Columns: 4
Duplicate Grain Records: 0
Null Key Records: 0
Unique Grain Records: 12,160



In [40]:
validate_mart(
    movie_production_country,
    ["Movie_ID", "Country_Code"],
    "Movie Production Country"
)

========== Movie Production Country ==========
Rows: 6,436
Columns: 4
Duplicate Grain Records: 0
Null Key Records: 0
Unique Grain Records: 6,436



In [41]:
validate_mart(
    movie_production_company,
    ["Movie_ID", "Production_Company_ID"],
    "Movie Production Company"
)

========== Movie Production Company ==========
Rows: 13,677
Columns: 4
Duplicate Grain Records: 0
Null Key Records: 0
Unique Grain Records: 13,677



                         SOURCE DATABASE
                               │
                               ▼
                         ┌───────────┐
                         │  MOVIES   │
                         └─────┬─────┘
                               │
             ┌─────────────────┼─────────────────┐
             │                 │                 │
             ▼                 ▼                 ▼
       MOVIE MASTER      ARTIST DIARY       FINANCIAL
       1 row/movie       Artist×Movie       1 row/movie
             │                 │
       ┌─────┼─────┐           ▼
       ▼     ▼     ▼      ARTIST PERFORMANCE
    Genre  Lang  Keyword      1 row/artist
       │     │     │
       ▼     ▼     ▼
   Separate bridge marts